In [17]:
import matplotlib.pyplot as plt
%matplotlib qt
import numpy as np
import pickle
import pandas as pd

from pathlib import Path
from src.models.sklearn_models import RandomForestModel, XGBModel
from src.models.keras_models import ConcatenatedModulesModel
from src.visualization.radar_chart import radial_chart


In [18]:


rf = RandomForestModel()
xgb = XGBModel()
dnn = ConcatenatedModulesModel("ConcatenatedDNN")
cnn = ConcatenatedModulesModel("ConcatenatedCNN")
bilstm = ConcatenatedModulesModel("ConcatenatedBiLSTM")

column_names = ['chlid', 'chl_a', 'chl_b', 'chc12', 'fucox', 'hxfcx', 'btfcx', 'diadi',
       'allox', 'diato', 'zeaxa', 'betac', 'perid']

graph_names = ['chlide', 'chla', 'chlb', 'chlc12', 'fuco', "hex", "but", 'diad',
       'allo', 'diato', 'zea', 'caro', 'peri']

translate_pigments = dict(zip(column_names, graph_names))

rename_models = {
    "ConcatenatedDNN": "DNN",
    "ConcatenatedCNN": "CNN",
    "ConcatenatedBiLSTM": "BiLSTM",
    "rf": "RF",
    "xgb": "XGB",
}

colors = ['#1E5799', '#AAAAAA', '#00A0B0', '#F0AD4E', '#C85217']

markers = ['^', 's', 'p', 'p', 'p']
linestyles = ['-', '--', ':', '-.', ':']

In [19]:
models = [cnn, dnn, bilstm, xgb, rf]
models = [cnn]

experiment_name = 'OLCI'
experiment_name = '13_wl_final'
# experiment_name = '5_wl_final'
# experiment_name = 5_wl_final
# experiment_name = 5_wl_final


In [20]:
def load_mean_metrics(path, models):
    paths_metrics = [p / 'metrics' for p in path.iterdir()]
    metrics_test = {model.name: { p.parent.name :pd.read_csv(p/ (model.name + '_test.csv'), index_col=0) for p in paths_metrics} for model in models}
    metrics_train = {model.name: { p.parent.name :pd.read_csv(p/ (model.name + '_train.csv'), index_col=0) for p in paths_metrics} for model in models}
    metrics_test_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_test.items()}
    metrics_train_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_train.items()}
    # metrics_test_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_test_mean.items()})
    # metrics_train_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_train_mean.items()})
    return metrics_train_mean, metrics_test_mean
 

In [21]:
experiment_name_from = "OLCI_sat_ft"
experiment_name_from = "OLCI"
model_label_from = r'$CNN$'
models = [ConcatenatedModulesModel("ConcatenatedCNN")]
path_experiments =  Path(f'../../experiments/{experiment_name_from}')
_, m = load_mean_metrics(path_experiments, models)
m1 = {model_label_from: m["ConcatenatedCNN"]}

In [22]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as path_effects

# ---------------------------------------------------------------------
# Data stored in m1
# ---------------------------------------------------------------------
model_label, df = next(iter(m1.items()))

required_metrics = ["R2", "MPE", "MAPE"]

for metric in required_metrics:
    if metric not in df.index:
        raise ValueError(f"{model_label} is missing metric: {metric}")

pigments = df.columns.to_numpy()

r2 = df.loc["R2", pigments].to_numpy(dtype=float)
mpe = df.loc["MPE", pigments].to_numpy(dtype=float) * 100
mape = df.loc["MAPE", pigments].to_numpy(dtype=float) * 100

# Sort by R² descending
order = np.argsort(r2)[::-1]

pigments = pigments[order]
r2 = r2[order]
mpe = mpe[order]
mape = mape[order]

# ---------------------------------------------------------------------
# Style
# ---------------------------------------------------------------------
plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "font.size": 16,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 15,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ---------------------------------------------------------------------
# Visual encodings
# ---------------------------------------------------------------------
cmap = mcolors.LinearSegmentedColormap.from_list(
    "r2_red_green",
    ["#E24B4A", "#F5C4B3", "#9FE1CB", "#1D9E75"]
)

norm = mcolors.Normalize(vmin=r2.min(), vmax=r2.max())

def bubble_size(mape_values):
    return 60 + 1400 * np.asarray(mape_values) / mape.max()

def bubble_size_legend(mape_values):
    return 40 + 650 * np.asarray(mape_values) / mape.max()

# ---------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9.5, 6.8))

ax.grid(True, color="#E6E6E6", linewidth=0.7)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_color("#CCCCCC")
    spine.set_linewidth(0.8)

# ---------------------------------------------------------------------
# Bubbles
# ---------------------------------------------------------------------
ax.scatter(
    r2,
    mpe,
    s=bubble_size(mape),
    c=r2,
    cmap=cmap,
    norm=norm,
    marker="o",
    edgecolors="black",
    linewidths=0.8,
    alpha=0.88,
    zorder=4,
)

# ---------------------------------------------------------------------
# Pigment labels
# ---------------------------------------------------------------------

# Manual offsets in points: (x_offset, y_offset)
# Adjust only problematic pigments here
# ['chlid', 'chl_a', 'chl_b', 'chc12', 'fucox', 'hxfcx', 'btfcx', 'diadi', 'allox', 'diato', 'zeaxa', 'betac', 'perid']
label_offsets = {
    # examples; adapt names to your actual translated/raw pigment names
    "chl_a":  (0, -20),
    "fucox":  (0,  10),
    "diadi":  (-10,  -30),
    "betac":  (-40, 0),
    "chc12":  (5,  10),
}

for i, pig in enumerate(pigments):
    pig_label = (
        translate_pigments[pig]
        if "translate_pigments" in globals() and pig in translate_pigments
        else pig
    )
    
    dx, dy = label_offsets.get(pig, (20, 0))
    ax.annotate(
        pig_label,
        xy=(r2[i], mpe[i]),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=15,
        fontweight="bold",
        color="black",
        ha="center",
        va="bottom",
        zorder=10,
        path_effects=[
            path_effects.Stroke(linewidth=2.5, foreground="white"),
            path_effects.Normal(),
        ],
    )

# ---------------------------------------------------------------------
# Axes
# ---------------------------------------------------------------------
ax.set_xlabel(r"$R^2$")
ax.set_ylabel("MPE [%]")
ax.tick_params(axis="both", labelsize=15)

# ---------------------------------------------------------------------
# MAPE-size legend
# ---------------------------------------------------------------------
mape_refs = [20, 150]

size_handles = [
    ax.scatter(
        [],
        [],
        s=bubble_size_legend(ref),
        marker="o",
        color="0.60",
        edgecolors="black",
        linewidths=0.8,
        alpha=0.85,
    )
    for ref in mape_refs
]

mape_leg = ax.legend(
    size_handles,
    [f"{ref}%" for ref in mape_refs],
    title="MAPE",
    loc="upper left",
    frameon=True,
    framealpha=0.95,
    edgecolor="#DDDDDD",
    labelspacing=0.5,
    borderpad=1.0,
    handletextpad=1.0,
    handlelength=1.3,
    handleheight=1.2,
)

mape_leg.get_frame().set_linewidth(0.8)

# ---------------------------------------------------------------------
# Colorbar
# ---------------------------------------------------------------------
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cb = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.04)
cb.set_label(r"$R^2$", fontsize=16, loc='top', rotation=0)
cb.outline.set_visible(False)
cb.ax.tick_params(labelsize=14, length=0)

# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------
plt.tight_layout()

out = "cnn_bubble_r2_mpe_mape_OLCI.jpg"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out)

Saved: cnn_bubble_r2_mpe_mape_OLCI.jpg


In [23]:
experiment_name_from = "OLCI_sat_ft"
experiment_name_from = "multi"
model_label_from = r'$CNN$'
models = [ConcatenatedModulesModel("ConcatenatedCNN")]
path_experiments =  Path(f'../../experiments/{experiment_name_from}')
_, m = load_mean_metrics(path_experiments, models)
m1 = {model_label_from: m["ConcatenatedCNN"]}

In [24]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as path_effects

# ---------------------------------------------------------------------
# Data stored in m1
# ---------------------------------------------------------------------
model_label, df = next(iter(m1.items()))

required_metrics = ["R2", "MPE", "MAPE"]

for metric in required_metrics:
    if metric not in df.index:
        raise ValueError(f"{model_label} is missing metric: {metric}")

pigments = df.columns.to_numpy()

r2 = df.loc["R2", pigments].to_numpy(dtype=float)
mpe = df.loc["MPE", pigments].to_numpy(dtype=float) * 100
mape = df.loc["MAPE", pigments].to_numpy(dtype=float) * 100

# Sort by R² descending
order = np.argsort(r2)[::-1]

pigments = pigments[order]
r2 = r2[order]
mpe = mpe[order]
mape = mape[order]

# ---------------------------------------------------------------------
# Style
# ---------------------------------------------------------------------
plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "font.size": 16,
    "axes.labelsize": 20,
    "axes.titlesize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 15,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ---------------------------------------------------------------------
# Visual encodings
# ---------------------------------------------------------------------
cmap = mcolors.LinearSegmentedColormap.from_list(
    "r2_red_green",
    ["#E24B4A", "#F5C4B3", "#9FE1CB", "#1D9E75"]
)

norm = mcolors.Normalize(vmin=r2.min(), vmax=r2.max())

def bubble_size(mape_values):
    return 60 + 1400 * np.asarray(mape_values) / mape.max()

def bubble_size_legend(mape_values):
    return 40 + 650 * np.asarray(mape_values) / mape.max()

# ---------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9.5, 6.8))

ax.grid(True, color="#E6E6E6", linewidth=0.7)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_color("#CCCCCC")
    spine.set_linewidth(0.8)

# ---------------------------------------------------------------------
# Bubbles
# ---------------------------------------------------------------------
ax.scatter(
    r2,
    mpe,
    s=bubble_size(mape),
    c=r2,
    cmap=cmap,
    norm=norm,
    marker="o",
    edgecolors="black",
    linewidths=0.8,
    alpha=0.88,
    zorder=4,
)

# ---------------------------------------------------------------------
# Pigment labels
# ---------------------------------------------------------------------

# Manual offsets in points: (x_offset, y_offset)
# Adjust only problematic pigments here
# ['chlid', 'chl_a', 'chl_b', 'chc12', 'fucox', 'hxfcx', 'btfcx', 'diadi', 'allox', 'diato', 'zeaxa', 'betac', 'perid']
label_offsets = {
    # examples; adapt names to your actual translated/raw pigment names
    "chl_a":  (0, -20),
    "fucox":  (0,  10),
    "diadi":  (-15,  -30),
    "betac":  (-40, 0),
    "chc12":  (5,  10),
}

for i, pig in enumerate(pigments):
    pig_label = (
        translate_pigments[pig]
        if "translate_pigments" in globals() and pig in translate_pigments
        else pig
    )
    
    dx, dy = label_offsets.get(pig, (20, 0))
    ax.annotate(
        pig_label,
        xy=(r2[i], mpe[i]),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=15,
        fontweight="bold",
        color="black",
        ha="center",
        va="bottom",
        zorder=10,
        path_effects=[
            path_effects.Stroke(linewidth=2.5, foreground="white"),
            path_effects.Normal(),
        ],
    )

# ---------------------------------------------------------------------
# Axes
# ---------------------------------------------------------------------
ax.set_xlabel(r"$R^2$")
ax.set_ylabel("MPE [%]")
ax.tick_params(axis="both", labelsize=15)

# ---------------------------------------------------------------------
# MAPE-size legend
# ---------------------------------------------------------------------
mape_refs = [20, 150]

size_handles = [
    ax.scatter(
        [],
        [],
        s=bubble_size_legend(ref),
        marker="o",
        color="0.60",
        edgecolors="black",
        linewidths=0.8,
        alpha=0.85,
    )
    for ref in mape_refs
]

mape_leg = ax.legend(
    size_handles,
    [f"{ref}%" for ref in mape_refs],
    title="MAPE",
    loc="upper left",
    frameon=True,
    framealpha=0.95,
    edgecolor="#DDDDDD",
    labelspacing=0.5,
    borderpad=1.0,
    handletextpad=1.0,
    handlelength=1.3,
    handleheight=1.2,
)

mape_leg.get_frame().set_linewidth(0.8)

# ---------------------------------------------------------------------
# Colorbar
# ---------------------------------------------------------------------
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cb = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.04)
cb.set_label(r"$R^2$", fontsize=16, loc='top', rotation=0)
cb.outline.set_visible(False)
cb.ax.tick_params(labelsize=14, length=0)

# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------
plt.tight_layout()

out = "cnn_bubble_r2_mpe_mape_multi.jpg"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out)

Saved: cnn_bubble_r2_mpe_mape_multi.jpg


In [61]:
experiment_name_from = "multi_sat_ft"
model_label_from = r'$CNN^*_{FT}$'
models = [ConcatenatedModulesModel("ConcatenatedCNN")]
path_experiments =  Path(f'../../experiments/{experiment_name_from}')
_, m = load_mean_metrics(path_experiments, models)
m1 = {model_label_from: m["ConcatenatedCNN"]}

experiment_name_to = "multi_sat_scratch"
model_label_to = r'$CNN^*_{SAT}$'
models = [ConcatenatedModulesModel("ConcatenatedCNN")]
path_experiments =  Path(f'../../experiments/{experiment_name_to}')
_, m = load_mean_metrics(path_experiments, models)
m2 = {model_label_to: m["ConcatenatedCNN"]}
       

In [62]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from matplotlib.legend_handler import HandlerBase
from matplotlib.lines import Line2D
import matplotlib.patheffects as path_effects
# ---------------------------------------------------------------------
# Models selected previously in m1 and m2
# ---------------------------------------------------------------------
model_from, df_from = next(iter(m2.items()))  # usually CNN_sat
model_to, df_to = next(iter(m1.items()))      # usually CNN_ft

# ---------------------------------------------------------------------
# Style
# ---------------------------------------------------------------------
plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "font.size": 16,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 14,
    "legend.title_fontsize": 14,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ---------------------------------------------------------------------
# Extract common pigments and metrics
# ---------------------------------------------------------------------
required_metrics = ["R2", "MPE", "MAPE"]

for metric in required_metrics:
    if metric not in df_from.index:
        raise ValueError(f"{model_from} is missing metric: {metric}")
    if metric not in df_to.index:
        raise ValueError(f"{model_to} is missing metric: {metric}")

pigments = df_from.columns.intersection(df_to.columns).to_numpy()

data = {
    model_from: {
        "r2":   df_from.loc["R2", pigments].to_numpy(dtype=float),
        "mpe":  df_from.loc["MPE", pigments].to_numpy(dtype=float) * 100,
        "mape": df_from.loc["MAPE", pigments].to_numpy(dtype=float) * 100,
        "marker": "s",
    },
    model_to: {
        "r2":   df_to.loc["R2", pigments].to_numpy(dtype=float),
        "mpe":  df_to.loc["MPE", pigments].to_numpy(dtype=float) * 100,
        "mape": df_to.loc["MAPE", pigments].to_numpy(dtype=float) * 100,
        "marker": "o",
    },
}

# Sort by target-model R² descending
order = np.argsort(data[model_to]["r2"])[::-1]
pigments = pigments[order]

for model in data:
    for metric in ["r2", "mpe", "mape"]:
        data[model][metric] = data[model][metric][order]

# ---------------------------------------------------------------------
# Visual encodings
# ---------------------------------------------------------------------
r2_all = np.concatenate([data[model_from]["r2"], data[model_to]["r2"]])
mape_all = np.concatenate([data[model_from]["mape"], data[model_to]["mape"]])

cmap = mcolors.LinearSegmentedColormap.from_list(
    "r2_red_green",
    ["#E24B4A", "#F5C4B3", "#9FE1CB", "#1D9E75"]
)

norm = mcolors.Normalize(vmin=r2_all.min(), vmax=r2_all.max())

def bubble_size(mape):
    return 60 + 1400 * np.asarray(mape) / mape_all.max()

# ---------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9.5, 6.8))

ax.grid(True, color="#E6E6E6", linewidth=0.7)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_color("#CCCCCC")
    spine.set_linewidth(0.8)


# ---------------------------------------------------------------------
# Scatter points
# ---------------------------------------------------------------------
for model in [model_from, model_to]:
    ax.scatter(
        data[model]["r2"],
        data[model]["mpe"],
        s=bubble_size(data[model]["mape"]),
        c=data[model]["r2"],
        cmap=cmap,
        norm=norm,
        marker=data[model]["marker"],
        edgecolors="black",
        linewidths=0.8,
        alpha=0.88,
        label=model,
        zorder=4,
    )

# ---------------------------------------------------------------------
# Arrows: model_from -> model_to
# ---------------------------------------------------------------------
for i in range(len(pigments)):
    ax.annotate(
        "",
        xy=(data[model_to]["r2"][i], data[model_to]["mpe"][i]),
        xytext=(data[model_from]["r2"][i], data[model_from]["mpe"][i]),
        arrowprops=dict(
            arrowstyle="->",
            color="0",
            lw=1.1,
            mutation_scale=12,
        ),
        zorder=4,
    )
# ---------------------------------------------------------------------
# Pigment labels
# ---------------------------------------------------------------------
for i, pig in enumerate(pigments):
    pig_label = (
        translate_pigments[pig]
        if "translate_pigments" in globals() and pig in translate_pigments
        else pig
    )

    ax.annotate(
        pig_label,
        xy=(data[model_to]["r2"][i], data[model_to]["mpe"][i]),
        xytext=(20, 0),
        textcoords="offset points",
        fontsize=13,
        fontweight="bold",
        color="black",
        ha="center",
        va="bottom",
        zorder=10,
        path_effects=[
        path_effects.Stroke(linewidth=2.5, foreground="white"),
        path_effects.Normal(),
    ],
    )

# ---------------------------------------------------------------------
# Axes
# ---------------------------------------------------------------------
ax.set_xlabel(r"$R^2$")
ax.set_ylabel("MPE [%]")
ax.tick_params(axis="both", labelsize=15)

# ---------------------------------------------------------------------
# Model legend
# ---------------------------------------------------------------------
h_from = ax.scatter(
    [],
    [],
    marker=data[model_from]["marker"],
    s=150,
    color="0.60",
    edgecolors="black",
    linewidths=0.8,
    alpha=0.85,
    label=model_from,
)

h_to = ax.scatter(
    [],
    [],
    marker=data[model_to]["marker"],
    s=150,
    color="0.60",
    edgecolors="black",
    linewidths=0.8,
    alpha=0.85,
    label=model_to,
)

model_leg = ax.legend(
    handles=[h_from, h_to],
    loc="upper right",
    frameon=True,
    framealpha=0.95,
    edgecolor="#DDDDDD",
    markerscale=1.0,
    scatterpoints=1,
    handletextpad=0.6,
    borderpad=0.6,
)

ax.add_artist(model_leg)

# ---------------------------------------------------------------------
# MAPE-size legend: square behind circle, same center
# ---------------------------------------------------------------------
mape_refs = [20, 40, 80]

def bubble_size_legend(mape):
    # Smaller than plot bubbles to avoid clipping inside legend frame
    return 40 + 650 * np.asarray(mape) / mape_all.max()


class HandlerSquareCircle(HandlerBase):
    def create_artists(
        self, legend, orig_handle, xdescent, ydescent,
        width, height, fontsize, trans
    ):
        s = orig_handle.get_sizes()[0]
        marker_size = np.sqrt(s)

        y = ydescent + height * 0.15

        # square slightly to the left
        x_square = xdescent + width * 0.50
        # circle on top, slightly shifted to the right
        x_circle = xdescent + width * 0.50

        square = Line2D(
            [x_square], [y],
            marker="s",
            markersize=marker_size,
            linestyle="None",
            markerfacecolor="0.60",
            markeredgecolor="black",
            markeredgewidth=0.8,
            alpha=0.85,
            transform=trans,
        )

        circle = Line2D(
            [x_circle], [y],
            marker="o",
            markersize=marker_size,
            linestyle="None",
            markerfacecolor="0.60",
            markeredgecolor="black",
            markeredgewidth=0.8,
            alpha=0.85,
            transform=trans,
        )

        return [square, circle]


size_handles = [
    ax.scatter(
        [],
        [],
        s=bubble_size_legend(ref),
        marker="o",
        color="0.60",
        edgecolors="black",
        linewidths=0.8,
        alpha=0.85,
    )
    for ref in mape_refs
]

mape_leg = ax.legend(
    size_handles,
    [f"{ref}%" for ref in mape_refs],
    title="MAPE",
    loc="upper left",
    frameon=True,
    framealpha=0.95,
    edgecolor="#DDDDDD",
    labelspacing=0.5,
    borderpad=1.0,
    handletextpad=1.0,
    handlelength=1.3,
    handleheight=1.2,
    handler_map={type(size_handles[0]): HandlerSquareCircle()},
)

mape_leg.get_frame().set_linewidth(0.8)

# ---------------------------------------------------------------------
# Colorbar
# ---------------------------------------------------------------------
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cb = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.04)
cb.set_label(r"$R^2$", fontsize=16)
cb.outline.set_visible(False)
cb.ax.tick_params(labelsize=14, length=0)

# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------
plt.tight_layout()

out = "cnn_comparison_r2_mpe_mape_multi.pdf"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out)

Saved: cnn_comparison_r2_mpe_mape_multi.pdf


In [74]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

pigments = ['chlide','chla','chlb','chlc12','fuco','hex','but','diad','allo','diato','zea','peri']

data = {
    'CNN_FT': {
        'r2':   [0.501,0.771,0.645,0.706,0.701,0.372,0.539,0.678,0.624,0.497,0.334,0.685],
        'mape': [98.8, 43.8, 70.6, 61.5, 71.2, 66.5, 61.5, 61.4,112.5, 61.7,119.0, 69.4],
        'mpe':  [51.5,  9.1, 33.5, 19.9, 17.2, 28.0, 27.9, 21.9, 71.6, 21.9, 82.8, 34.3],
    },
    'CNN_SAT': {
        'r2':   [0.429,0.705,0.560,0.633,0.657,0.113,0.271,0.621,0.488,0.473,0.230,0.562],
        'mape': [125.3,55.6, 87.6, 79.4, 97.8, 91.9, 80.5, 78.3,139.7, 71.1,140.0, 91.5],
        'mpe':  [77.0, 21.3, 51.7, 37.3, 45.4, 49.9, 41.2, 40.0, 98.7, 32.9, 99.1, 52.6],
    },
}

# sort by CNN_FT R² descending
order = sorted(range(len(pigments)), key=lambda i: data['CNN_FT']['r2'][i], reverse=True)
pigments = [pigments[i] for i in order]
for model in data:
    for k in data[model]:
        data[model][k] = [data[model][k][i] for i in order]

for model in data:
    for k in data[model]:
        data[model][k] = np.array(data[model][k])

# global MPE max for consistent bubble scaling
mape_all = np.concatenate([data['CNN_FT']['mape'], data['CNN_SAT']['mape']])
mape_max = mape_all.max()

def bubble_size(mape): return (mape / mape_max) * 1400 + 60

# colour ramp by R² (shared across both models)
import matplotlib.colors as mcolors
cmap = mcolors.LinearSegmentedColormap.from_list('rg', ['#E24B4A','#F5C4B3','#9FE1CB','#1D9E75'])
r2_all = np.concatenate([data['CNN_FT']['r2'], data['CNN_SAT']['r2']])
norm   = mcolors.Normalize(vmin=r2_all.min(), vmax=r2_all.max())

def r2_color(r2_arr):
    return [cmap(norm(v)) for v in r2_arr]

# ── figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9.5, 6.8))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

ax.grid(True, color='#E8E7E1', linewidth=0.6, zorder=0)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_color('#D3D1C7')
    spine.set_linewidth(0.6)

# connecting arrows (SAT → FT)
for i in range(len(pigments)):
    x0, y0 = data['CNN_SAT']['r2'][i], data['CNN_SAT']['mpe'][i]
    x1, y1 = data['CNN_FT']['r2'][i],  data['CNN_FT']['mpe'][i]
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color='#B4B2A9',
                                lw=0.9, mutation_scale=10),
                zorder=5)

# bubbles
ax.scatter(data['CNN_SAT']['r2'], data['CNN_SAT']['mpe'],
           s=bubble_size(data['CNN_SAT']['mape']),
           c=r2_color(data['CNN_SAT']['r2']),
           edgecolors='white', linewidths=0.8,
           zorder=3, alpha=0.85, marker='s')

ax.scatter(data['CNN_FT']['r2'], data['CNN_FT']['mpe'],
           s=bubble_size(data['CNN_FT']['mape']),
           c=r2_color(data['CNN_FT']['r2']),
           edgecolors='white', linewidths=0.8,
           zorder=4, alpha=0.88, marker='o')

# pigment labels (follow CNN_FT position)
label_offsets = {
    'chla':   ( 0.004,  5), 'chlc12': ( 0.003,  5), 'fuco':  ( 0.003,  5),
    'diad':   ( 0.003,  5), 'allo':   ( 0.003,  5), 'chlb':  ( 0.003,  5),
    'but':    ( 0.003,  5), 'diato':  ( 0.003,  5), 'peri':  ( 0.003,  5),
    'chlide': (-0.005, -9), 'zea':    ( 0.003,  5), 'hex':   (-0.005, -9),
}
for i, pig in enumerate(pigments):
    dx, dy = label_offsets.get(pig, (0.003, 5))
    ax.annotate(pig,
                xy=(data['CNN_FT']['r2'][i], data['CNN_FT']['mpe'][i]),
                xytext=(data['CNN_FT']['r2'][i] + dx,
                        data['CNN_FT']['mpe'][i] + dy),
                fontsize=8, color='#444441', ha='left', va='bottom')

# ── axes labels ───────────────────────────────────────────────────────────────
ax.set_xlabel('R²', fontsize=10, color='#2C2C2A', labelpad=8)
ax.set_ylabel('MPE (%)', fontsize=10, color='#2C2C2A', labelpad=8)
ax.tick_params(colors='#5F5E5A', labelsize=8.5)

# ── colorbar (R²) ─────────────────────────────────────────────────────────────
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.03)
cb.set_label('R²', fontsize=9, color='#2C2C2A')
cb.ax.tick_params(labelsize=8, length=0)
cb.outline.set_visible(False)

# ── marker legend (model) ─────────────────────────────────────────────────────
mid_color = cmap(0.5)
h_ft  = ax.scatter([], [], marker='o', color=mid_color, s=80, label='CNN$_{FT}$')
h_sat = ax.scatter([], [], marker='s', color=mid_color, s=80, label='CNN$_{SAT}$')
model_leg = ax.legend(handles=[h_ft, h_sat],
                      title='Model', title_fontsize=8.5,
                      fontsize=8.5, frameon=True, loc='upper right',
                      framealpha=0.92, edgecolor='#D3D1C7')
model_leg.get_frame().set_linewidth(0.5)
ax.add_artist(model_leg)

# ── bubble size legend — 3 rows, circle + square per row ─────────────────────
from matplotlib.legend_handler import HandlerTuple
handles, labels = [], []
for mape_ref in [20, 40, 80]:
    s = bubble_size(mape_ref)
    h_c = ax.scatter([], [], s=s, marker='o', color='#B4B2A9',
                     edgecolors='white', linewidths=0.6, alpha=0.85)
    h_s = ax.scatter([], [], s=s, marker='s', color='#B4B2A9',
                     edgecolors='white', linewidths=0.6, alpha=0.85)
    handles.append((h_c, h_s))
    labels.append(f'MAPE = {mape_ref}%')

size_leg = ax.legend(handles, labels,
                     handler_map={tuple: HandlerTuple(ndivide=2, pad=7.5)},
                     title='Bubble size → MAPE   ○ CNN$_{FT}$   □ CNN$_{SAT}$',
                     title_fontsize=7.5,
                     fontsize=7.5, frameon=True, loc='lower left',
                     framealpha=0.92, edgecolor='#D3D1C7',
                     handlelength=5, handletextpad=0.8, labelspacing=0.6)
size_leg.get_frame().set_linewidth(0.5)

ax.set_title('CNN$_{FT}$ vs CNN$_{SAT}$ — R² vs MPE  ·  bubble size encodes MAPE\n'
             'Arrows point from CNN$_{SAT}$ to CNN$_{FT}$',
             fontsize=10, color='#2C2C2A', pad=12)

plt.tight_layout()

out = 'tmp.pdf'
with PdfPages(out) as pdf:
    pdf.savefig(fig, bbox_inches='tight', dpi=200)
plt.close()
print("Done:", out)


Done: tmp.pdf


In [68]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ---------------------------------------------------------------------
# Models selected previously in m1 and m2
# ---------------------------------------------------------------------
model_from, df_from = next(iter(m2.items()))  # e.g. CNN_sat
model_to, df_to = next(iter(m1.items()))      # e.g. CNN_ft

# ---------------------------------------------------------------------
# Style
# ---------------------------------------------------------------------
plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "font.size": 18,
    "axes.labelsize": 22,
    "axes.titlesize": 20,
    "xtick.labelsize": 17,
    "ytick.labelsize": 17,
    "legend.fontsize": 16,
    "legend.title_fontsize": 16,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ---------------------------------------------------------------------
# Extract common pigments and metrics
# ---------------------------------------------------------------------
required_metrics = ["MAPE", "MAE", "R2"]

for metric in required_metrics:
    if metric not in df_from.index:
        raise ValueError(f"{model_from} is missing metric: {metric}")
    if metric not in df_to.index:
        raise ValueError(f"{model_to} is missing metric: {metric}")

pigments = df_from.columns.intersection(df_to.columns).to_numpy()

data = {
    model_from: {
        "mape": df_from.loc["MAPE", pigments].to_numpy(dtype=float) * 100,
        "mae":  df_from.loc["MAE", pigments].to_numpy(dtype=float),
        "r2":   df_from.loc["R2", pigments].to_numpy(dtype=float),
    },
    model_to: {
        "mape": df_to.loc["MAPE", pigments].to_numpy(dtype=float) * 100,
        "mae":  df_to.loc["MAE", pigments].to_numpy(dtype=float),
        "r2":   df_to.loc["R2", pigments].to_numpy(dtype=float),
    },
}

# Sort by target-model R² descending
order = np.argsort(data[model_to]["r2"])[::-1]
pigments = pigments[order]

for model in data:
    for metric in data[model]:
        data[model][metric] = data[model][metric][order]

# Optional pigment-name translation
pigment_labels = [
    translate_pigments[p] if "translate_pigments" in globals() and p in translate_pigments else p
    for p in pigments
]

# ---------------------------------------------------------------------
# Visual encodings
# ---------------------------------------------------------------------
r2_all = np.concatenate([data[model_from]["r2"], data[model_to]["r2"]])
mape_all = np.concatenate([data[model_from]["mape"], data[model_to]["mape"]])
mae_all = np.concatenate([data[model_from]["mae"], data[model_to]["mae"]])

cmap = mcolors.LinearSegmentedColormap.from_list(
    "r2_red_green",
    ["#E24B4A", "#F5C4B3", "#9FE1CB", "#1D9E75"]
)

norm = mcolors.Normalize(vmin=r2_all.min(), vmax=r2_all.max())

mae_max = mae_all.max()

def bubble_size(mae):
    diameter = 9 + 24 * np.asarray(mae) / mae_max
    return diameter**2

# ---------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(13.5, 7.5))

x = np.arange(len(pigments))
dx = 0.18

x_from = x - dx
x_to = x + dx

ax.grid(True, axis="y", color="#E6E6E6", linewidth=0.8)
ax.grid(False, axis="x")
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_color("#CCCCCC")
    spine.set_linewidth(0.8)

# ---------------------------------------------------------------------
# Arrows: model_from -> model_to
# ---------------------------------------------------------------------
for i in range(len(pigments)):
    ax.annotate(
        "",
        xy=(x_to[i], data[model_to]["mape"][i]),
        xytext=(x_from[i], data[model_from]["mape"][i]),
        arrowprops=dict(
            arrowstyle="->",
            color="0.35",
            lw=1.3,
            mutation_scale=13,
            shrinkA=5,
            shrinkB=5,
        ),
        zorder=3,
    )

# ---------------------------------------------------------------------
# Scatter points
# ---------------------------------------------------------------------
ax.scatter(
    x_from,
    data[model_from]["mape"],
    s=bubble_size(data[model_from]["mae"]),
    c=data[model_from]["r2"],
    cmap=cmap,
    norm=norm,
    marker="s",
    edgecolors="white",
    linewidths=0.9,
    alpha=0.90,
    label=model_from,
    zorder=4,
)

ax.scatter(
    x_to,
    data[model_to]["mape"],
    s=bubble_size(data[model_to]["mae"]),
    c=data[model_to]["r2"],
    cmap=cmap,
    norm=norm,
    marker="o",
    edgecolors="white",
    linewidths=0.9,
    alpha=0.92,
    label=model_to,
    zorder=5,
)

# ---------------------------------------------------------------------
# Axes
# ---------------------------------------------------------------------
ax.set_ylabel("MAPE [%]")
ax.set_xlabel("Pigment")

ax.set_xticks(x)
ax.set_xticklabels(pigment_labels, rotation=45, ha="right")

ypad = 0.08 * (mape_all.max() - mape_all.min())
ax.set_ylim(max(0, mape_all.min() - ypad), mape_all.max() + ypad)

ax.set_xlim(-0.6, len(pigments) - 0.4)

ax.set_title(
    f"{model_to} vs {model_from} performance by pigment",
    pad=14,
)

ax.tick_params(axis="both", labelsize=17)

# ---------------------------------------------------------------------
# Model legend
# ---------------------------------------------------------------------
model_leg = ax.legend(
    title="Model",
    loc="upper left",
    frameon=True,
    framealpha=0.95,
    edgecolor="#DDDDDD",
)

ax.add_artist(model_leg)

# ---------------------------------------------------------------------
# MAE-size legend
# ---------------------------------------------------------------------
mae_refs = np.linspace(mae_all.min(), mae_all.max(), 3)

size_handles = [
    ax.scatter(
        [],
        [],
        s=bubble_size(ref),
        color="0.60",
        edgecolors="white",
        linewidths=0.9,
        alpha=0.90,
    )
    for ref in mae_refs
]

ax.legend(
    size_handles,
    [f"{ref:.2f}" for ref in mae_refs],
    title="MAE",
    loc="upper right",
    frameon=True,
    framealpha=0.95,
    edgecolor="#DDDDDD",
    labelspacing=1.1,
)

# ---------------------------------------------------------------------
# Colorbar: R²
# ---------------------------------------------------------------------
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cb = fig.colorbar(sm, ax=ax, pad=0.015, fraction=0.035)
cb.set_label(r"$R^2$", fontsize=20)
cb.outline.set_visible(False)
cb.ax.tick_params(labelsize=16, length=0)

# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------
plt.tight_layout()

out = "cnn_comparison_by_pigment_mape_mae_r2.pdf"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out)

Saved: cnn_comparison_by_pigment_mape_mae_r2.pdf


In [7]:
# select values to plot (R2 here)
r2_mets = pd.DataFrame(ds.loc['R2',:].rename(model_name) for model_name, ds in metrics_test_mean.items())

In [8]:
r2_mets = r2_mets.sort_values(axis=1, by='ConcatenatedCNN', ascending=False)
r2_mets = r2_mets.rename(dict(zip(column_names, graph_names)), axis=1)
r2_mets = r2_mets.rename(rename_models, axis=0)

In [9]:
# pigments = np.array(['chlide_a', 'chla   ', 'chlb      ', 'chlc1+c2              ', 'fucox           ', "19'hxfcx               ", 
#                      "19'btfcx         ", "             diadino", "        allox", "           diatox", "            zeaxan", "              beta_car",
#                      "               peridinin"])




In [10]:
radial_chart(r2_mets.values, r2_mets.columns, r2_mets.index, colors=colors, markers=markers, linestyles=linestyles, title='', save=None)

In [11]:
r2_mets

,chla,chlc12,caro,fuco,diad,peri,chlb,diato,chlide,hex,allo,but,zea
CNN,0.927271,0.924595,0.915100,0.898419,0.882132,0.836779,0.791856,0.770290,0.748999,0.669882,0.649530,0.549060,0.548819
DNN,0.871697,0.883277,0.845381,0.872509,0.848491,0.811355,0.678291,0.690064,0.749485,0.597398,0.566663,0.297228,0.293975
BiLSTM,0.894677,0.909448,0.883202,0.882325,0.882208,0.812700,0.724412,0.743249,0.751519,0.715847,0.610763,0.458672,0.409853
XGB,0.912795,0.920872,0.889833,0.896103,0.860384,0.854783,0.770071,0.743582,0.664891,0.551984,0.555442,0.473469,0.429347
RF,0.898049,0.914259,0.883112,0.901023,0.873083,0.853038,0.739886,0.719772,0.745741,0.620450,0.574591,0.416496,0.460731


In [12]:
#  fine tunning model vs scratch:

experiment_name_cnn = 'OLCI_sat_scratch'
experiment_name_cnn_ast = 'OLCI_sat_ft'

path_experiments_cnn =  Path(f'../../experiments/{experiment_name_cnn}')
paths_metrics_cnn = [p / 'metrics' for p in path_experiments_cnn.iterdir()]

path_experiments_cnn_ast =  Path(f'../../experiments/{experiment_name_cnn_ast}')
paths_metrics_cnn_ast = [p / 'metrics' for p in path_experiments_cnn_ast.iterdir()]

# Load metrics:
metrics_test = {
    r'$CNN_{ft}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_test.csv'), index_col=0) for p in paths_metrics_cnn_ast},
    r'$CNN_{sat}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_test.csv'), index_col=0) for p in paths_metrics_cnn},
               }
metrics_train = {r'$CNN_{ft}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_train.csv'), index_col=0) for p in paths_metrics_cnn_ast},
                 r'$CNN_{sat}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_train.csv'), index_col=0) for p in paths_metrics_cnn},
               }        

In [13]:
metrics_test_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_test.items()}
metrics_train_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_train.items()}
metrics_test_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_test_mean.items()})
metrics_train_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_train_mean.items()})

In [14]:
# select values to plot (R2 here)
r2_mets = pd.DataFrame(ds.loc['R2',:].rename(model_name) for model_name, ds in metrics_test_mean.items())

In [15]:
r2_mets = r2_mets.sort_values(axis=1, by=r'$CNN_{ft}$', ascending=False)
r2_mets = r2_mets.rename(dict(zip(column_names, graph_names)), axis=1)
r2_mets = r2_mets.rename(rename_models, axis=0)

In [16]:
r2_mets

,chla,chlc12,fuco,peri,diad,chlb,allo,but,chlide,diato,hex,zea
$CNN_{ft}$,0.770875,0.706369,0.701201,0.685386,0.677783,0.645425,0.623559,0.539100,0.501054,0.496587,0.371794,0.333607
$CNN_{sat}$,0.704584,0.633023,0.657017,0.561831,0.620584,0.560095,0.487923,0.270706,0.429194,0.472730,0.112703,0.230268


In [17]:
r2_mets

,chla,chlc12,fuco,peri,diad,chlb,allo,but,chlide,diato,hex,zea
$CNN_{ft}$,0.770875,0.706369,0.701201,0.685386,0.677783,0.645425,0.623559,0.539100,0.501054,0.496587,0.371794,0.333607
$CNN_{sat}$,0.704584,0.633023,0.657017,0.561831,0.620584,0.560095,0.487923,0.270706,0.429194,0.472730,0.112703,0.230268


In [18]:
colors = ['#1E5799', '#00A0B0']

radial_chart(r2_mets.values, r2_mets.columns, r2_mets.index, colors=colors, markers=markers, linestyles=linestyles, title='', save=None)

In [15]:
metrics_test_mean_mean

,$CNN_{ft}$,$CNN_{sat}$
MAE,0.111245,0.126751
MAPE,0.748279,0.949031
ME,-0.055999,-0.061957
MPE,0.349705,0.539231
MSE,0.123596,0.154226
R2,0.587728,0.478388


In [16]:
#  5 vs 11:

experiment_name_cnn = 'multi'
experiment_name_cnn_ast = 'OLCI'

path_experiments_cnn =  Path(f'../../experiments/{experiment_name_cnn}')
paths_metrics_cnn = [p / 'metrics' for p in path_experiments_cnn.iterdir()]

path_experiments_cnn_ast =  Path(f'../../experiments/{experiment_name_cnn_ast}')
paths_metrics_cnn_ast = [p / 'metrics' for p in path_experiments_cnn_ast.iterdir()]

# Load metrics:
metrics_test = {
    r'$CNN$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_test.csv'), index_col=0) for p in paths_metrics_cnn_ast},
    r'$CNN^*$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_test.csv'), index_col=0) for p in paths_metrics_cnn},
               }
metrics_train = {r'$CNN$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_train.csv'), index_col=0) for p in paths_metrics_cnn_ast},
                 r'$CNN^*$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_train.csv'), index_col=0) for p in paths_metrics_cnn},
               }        

In [17]:
metrics_test_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_test.items()}
metrics_train_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_train.items()}
metrics_test_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_test_mean.items()})
metrics_train_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_train_mean.items()})

# select values to plot (R2 here)
r2_mets = pd.DataFrame(ds.loc['R2',:].rename(model_name) for model_name, ds in metrics_test_mean.items())

r2_mets = r2_mets.sort_values(axis=1, by=r'$CNN$', ascending=False)
r2_mets = r2_mets.rename(dict(zip(column_names, graph_names)), axis=1)
r2_mets = r2_mets.rename(rename_models, axis=0)

r2_mets

,chlc12,chla,fuco,betac,diadino,peri,chlb,diato,chlide,19'hxfuco,allo,zea,19'btfuco
$CNN$,0.929390,0.916206,0.908330,0.907215,0.891906,0.838785,0.783394,0.782221,0.777933,0.681189,0.608167,0.543872,0.518867
$CNN^*$,0.925064,0.907958,0.908219,0.893581,0.888608,0.850047,0.764915,0.752506,0.776724,0.686888,0.603514,0.541505,0.499856


In [18]:
metrics_test_mean_mean

,$CNN$,$CNN^*$
MAE,0.043647,0.044507
MAPE,0.545656,0.514315
ME,-0.013982,-0.012873
MPE,0.259788,0.217597
MSE,0.063260,0.065166
R2,0.775960,0.769183


In [19]:
colors = ['#1E5799', '#00A0B0']

radial_chart(r2_mets.values, r2_mets.columns, r2_mets.index, colors=colors, markers=markers, linestyles=linestyles, title='', save=None)

In [19]:
#  fine tunning model vs scratch 5:

experiment_name_cnn = 'multi_sat_scratch'
experiment_name_cnn_ast = 'multi_sat_ft'

path_experiments_cnn =  Path(f'../../experiments/{experiment_name_cnn}')
paths_metrics_cnn = [p / 'metrics' for p in path_experiments_cnn.iterdir()]

path_experiments_cnn_ast =  Path(f'../../experiments/{experiment_name_cnn_ast}')
paths_metrics_cnn_ast = [p / 'metrics' for p in path_experiments_cnn_ast.iterdir()]

# Load metrics:
metrics_test = {
    r'$CNN^*_{ft}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_test.csv'), index_col=0) for p in paths_metrics_cnn_ast},
    r'$CNN^*_{sat}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_test.csv'), index_col=0) for p in paths_metrics_cnn},
               }
metrics_train = {r'$CNN^*_{ft}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_train.csv'), index_col=0) for p in paths_metrics_cnn_ast},
                 r'$CNN^*_{sat}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_train.csv'), index_col=0) for p in paths_metrics_cnn},
               } 

metrics_test_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_test.items()}
metrics_train_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_train.items()}
metrics_test_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_test_mean.items()})
metrics_train_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_train_mean.items()})

# select values to plot (R2 here)
r2_mets = pd.DataFrame(ds.loc['R2',:].rename(model_name) for model_name, ds in metrics_test_mean.items())

r2_mets = r2_mets.sort_values(axis=1, by=r'$CNN^*_{ft}$', ascending=False)
r2_mets = r2_mets.rename(dict(zip(column_names, graph_names)), axis=1)
r2_mets = r2_mets.rename(rename_models, axis=0)

colors = ['#1E5799', '#00A0B0']

radial_chart(r2_mets.values, r2_mets.columns, r2_mets.index, colors=colors, markers=markers, linestyles=linestyles, title='', save=None)

In [5]:
data = pd.read_csv("../../data/datasets/hplc_world/hplc_multi.csv", low_memory=False)

In [15]:
data["chlide_a[mg*m^3]"].plot.hist(bins=100)

<Axes: ylabel='Frequency'>